# FinTrust LLM forecaster: self-hosted run on Kaggle

Runs the LLM agent of `paper/experiments/llm/PREREGISTRATION.md` on Kaggle's free GPUs. Nothing runs on your laptop.

**One-time setup**
1. On huggingface.co, open [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) and
   [Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) and accept the licence (approval usually takes minutes).
   Then create a **read** token under Settings > Access Tokens.
2. On kaggle.com, verify your phone number (needed for GPU and internet).
3. Create a notebook, then File > Import Notebook and upload this file.
4. Right-hand panel > Session options: **Accelerator = GPU T4 x2**, **Internet = on**.
5. Add-ons > Secrets: add `HF_TOKEN` with the token from step 1 and tick it for this notebook.
6. Optional but better for reproducibility: upload the paper's `nse_daily.csv` as a private Kaggle Dataset and add it
   to the notebook (Add Input). Without it the notebook downloads a fresh copy from Yahoo Finance.

**Run**: Save Version > *Save & Run All (Commit)*. It runs in the background (you can close the browser) and
should take roughly 1 to 3 hours (estimate). When it finishes, open the version's **Output** tab and download
`llm_outputs.zip`. Unzip it into `paper/experiments/llm/outputs/` and run `python run_llm.py`.

In [ ]:
import os, sys, glob, subprocess, hashlib, shutil, time
REPO = 'https://github.com/IAbdullahSlash/fintrust.git'
BRANCH = 'claude/project-thread-k1tvaj'          # falls back to the default branch once this one is merged
if not os.path.exists('fintrust'):
    if subprocess.call(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, 'fintrust']) != 0:
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO, 'fintrust'])
print(subprocess.check_output(['git', '-C', 'fintrust', 'log', '-1', '--format=%H %s']).decode())

In [ ]:
!pip -q install -U "transformers>=4.45" accelerate yfinance
import torch; print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Price data: the paper's copy if attached as a Kaggle Dataset, otherwise a fresh Yahoo Finance download
found = glob.glob('/kaggle/input/**/nse_daily.csv', recursive=True)
if found: PRICES = found[0]
else:
    subprocess.check_call([sys.executable, 'fintrust/data/download_nse.py']); PRICES = 'fintrust/data/raw/nse_daily.csv'
print(PRICES, hashlib.sha256(open(PRICES, 'rb').read()).hexdigest())

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login; login(token=os.environ['HF_TOKEN'])
sys.path.insert(0, 'fintrust/paper/experiments/llm')
import llm_forecaster as L
REQ, OUT = '/kaggle/working/requests', '/kaggle/working/outputs'
counts = L.build(PRICES, REQ, label_csv='fintrust/paper/experiments/fair/nse_vol_forecasts.csv')

In [ ]:
# Main model: Llama 3.1 8B Instruct (knowledge cutoff Dec 2023), fp16 across both T4s, greedy decoding
be = L.HFBackend('meta-llama/Llama-3.1-8B-Instruct', max_batch=16)
print('revision', be.revision)
t0 = time.time(); p = L.run(be, f'{REQ}/clean.jsonl', OUT, 'llama31_8b', limit=32)      # pilot: timing and sample answers
print(f'{time.time() - t0:.1f}s for 32'); print(open(p).read()[:1500])

In [ ]:
for cond in ['clean', 'clean_pre', 'named', 'named_pre', 'news_alarm', 'news_calm', 'persona', 'rationale']:
    t0 = time.time(); L.run(be, f'{REQ}/{cond}.jsonl', OUT, 'llama31_8b'); print(cond, f'{(time.time() - t0) / 60:.1f} min', flush=True)

In [ ]:
# F4 silent model swap: a smaller model answers the same 2025 prompts
del be; import gc; gc.collect(); torch.cuda.empty_cache()
be = L.HFBackend('meta-llama/Llama-3.2-1B-Instruct', max_batch=32)
L.run(be, f'{REQ}/clean.jsonl', OUT, 'llama32_1b', start='2025-01-01')

In [ ]:
import json, collections
for f in sorted(glob.glob(f'{OUT}/*.jsonl')):
    rows = [json.loads(l) for l in open(f)]
    print(os.path.basename(f), len(rows), 'parse failures:', sum(not r['parse_ok'] for r in rows),
          'mean p: %.3f' % (sum(r['p'] for r in rows) / len(rows)))
shutil.make_archive('/kaggle/working/llm_outputs', 'zip', OUT); print('download llm_outputs.zip from the Output tab')